# 01 Presentation: OpenICU concepts → YAIB dynamic wide table

Build `openicu_dyn_presentation.parquet` from already stay/time-indexed OpenICU concept parquets.

This notebook does **not** use RICU and does **not** read `icustays.csv.gz`. The concept parquets are expected to already contain `stay_id`, induced integer `time`, and `numeric_value`.


In [ ]:
from pathlib import Path
import polars as pl

from openicu_yaib.transform import build_dynamic_table
from openicu_yaib.concepts import DYNAMIC_VARS

## Paths

Adjust these paths for your machine.

In [ ]:
CONCEPT_ROOT = Path("~/output/OpenICU.example/project/workspace/concept")
OUTPUT = Path("../output/openicu_dyn_presentation.parquet")

DATASET = "mimic-iv"
VERSION = "1.0.0"

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
OUTPUT


## Build dynamic table

Minimal path: read concept files from `CONCEPT_ROOT`, select the configured dynamic concepts, aggregate duplicate `(stay_id, time)` keys if they exist, and full-join everything into the YAIB-style wide format.

No RICU metadata, no ICU stay table, no timestamp remapping, no grid construction.


In [ ]:
lf = build_dynamic_table(
    concept_root=CONCEPT_ROOT,
    dataset=DATASET,
    version=VERSION,
    dynamic_vars=DYNAMIC_VARS,
    aggregation_mode="mean",
    include_grid=False,
    missing_concepts="warn",
)

lf.sink_parquet(OUTPUT)
OUTPUT


## Quick check

In [ ]:
df = pl.scan_parquet(OUTPUT)

df.select([
    pl.len().alias("n_rows"),
    pl.col("stay_id").n_unique().alias("n_stays"),
    pl.col("time").min().alias("min_time"),
    pl.col("time").max().alias("max_time"),
]).collect()

In [ ]:
df.collect()